# Module 4 — Single-Cell RNA-seq: QC → Clustering → Annotation → Function
### 15-minute live demo

This is a condensed, **demo-only** version of the full Module 4 notebook. It walks through the four core steps of a single-cell analysis on a COVID-19 BALF sample (GEO **GSE145926**, sample C141):

1. **Quality control** — filter low-quality cells
2. **Clustering** — group cells by expression
3. **Cell-type annotation** — label the clusters
4. **Functional analysis** — GO enrichment of cluster markers

> **Run the setup block below *before* the session.** It installs packages, downloads the data, and computes everything slow (normalization, PCA, UMAP, clustering, markers, reference annotation). Once it finishes, every cell in the demo is either instant or just draws a figure from the pre-computed object.

---
## ⚙️ Setup — run this once before the demo (takes several minutes)

Collapse this cell during the talk. It reproduces the full pipeline so the object `sampleA` arrives at the demo already normalized, reduced, clustered, and annotated.

In [ ]:
# --- Packages ---------------------------------------------------------------
# Install any that are missing, then load. (Safe to re-run.)
pkgs <- c("Seurat", "tidyverse", "clusterProfiler", "celldex", "SingleR",
          "org.Hs.eg.db", "pheatmap")
for (p in pkgs) if (!requireNamespace(p, quietly = TRUE)) {
  if (p %in% c("clusterProfiler","celldex","SingleR","org.Hs.eg.db")) {
    if (!requireNamespace("BiocManager", quietly = TRUE)) install.packages("BiocManager")
    BiocManager::install(p, update = FALSE, ask = FALSE)
  } else install.packages(p)
}
invisible(lapply(pkgs, library, character.only = TRUE))

# --- Download data (GEO GSE145926, sample C141) -----------------------------
if (!file.exists("scRNAseq/GSM4339769_C141_filtered_feature_bc_matrix.h5")) {
  dir.create("scRNAseq", showWarnings = FALSE)
  download.file(
    "https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4339nnn/GSM4339769/suppl/GSM4339769_C141_filtered_feature_bc_matrix.h5",
    "scRNAseq/GSM4339769_C141_filtered_feature_bc_matrix.h5", mode = "wb")
}

# --- Build the object through clustering ------------------------------------
sc      <- Read10X_h5("scRNAseq/GSM4339769_C141_filtered_feature_bc_matrix.h5")
sampleA <- CreateSeuratObject(counts = sc, project = "sampleA",
                              min.cells = 3, min.features = 200)
sampleA[["percent.mt"]] <- PercentageFeatureSet(sampleA, pattern = "^MT-")

# NOTE: QC filtering is applied live in the demo (Step 1), not here,
# so the 'before/after' is visible. Everything downstream is pre-computed
# on the FILTERED object below.
sampleA <- subset(sampleA,
                  nCount_RNA > 500 & nFeature_RNA < 7000 & percent.mt < 10)

sampleA <- NormalizeData(sampleA, normalization.method = "LogNormalize", scale.factor = 10000)
sampleA <- FindVariableFeatures(sampleA, selection.method = "vst", nfeatures = 2000)
sampleA <- ScaleData(sampleA)
sampleA <- RunPCA(sampleA, features = VariableFeatures(sampleA))
DefaultAssay(sampleA) <- "RNA"
sampleA <- RunUMAP(sampleA, dims = 1:15, verbose = FALSE)
sampleA <- FindNeighbors(sampleA, dims = 1:20)
sampleA <- FindClusters(sampleA, resolution = 0.01)  # low res -> a few clean clusters

# --- Cluster markers (used in clustering + GO steps) ------------------------
sampleA.markers <- FindAllMarkers(sampleA, only.pos = TRUE,
                                  min.pct = 0.25, logfc.threshold = 0.25)

# --- Reference-based annotation (SingleR) -----------------------------------
ref     <- celldex::BlueprintEncodeData()
my.sce  <- as.SingleCellExperiment(sampleA)
pred    <- SingleR(my.sce, ref = ref, labels = ref$label.main)

# --- Gene mapping for GO (SYMBOL -> ENTREZID) -------------------------------
my.genes <- sampleA.markers %>%
  filter(abs(avg_log2FC) > 1, p_val_adj < 0.10) %>%
  dplyr::select(gene) %>% pull()
my.map   <- bitr(my.genes, fromType = "SYMBOL", toType = "ENTREZID", OrgDb = "org.Hs.eg.db")

cat("Setup complete. Cells:", ncol(sampleA),
    "| Clusters:", length(levels(sampleA)),
    "| Markers:", nrow(sampleA.markers), "\n")


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘bitops’, ‘dotCall64’, ‘lazyeval’, ‘gtools’, ‘caTools’, ‘plyr’, ‘tensor’, ‘BH’, ‘sitmo’, ‘sp’, ‘spam’, ‘globals’, ‘listenv’, ‘parallelly’, ‘zoo’, ‘crosstalk’, ‘RcppTOML’, ‘here’, ‘gplots’, ‘reshape2’, ‘gridExtra’, ‘RcppArmadillo’, ‘spatstat.data’, ‘spatstat.univar’, ‘spatstat.random’, ‘spatstat.utils’, ‘spatstat.sparse’, ‘goftest’, ‘abind’, ‘deldir’, ‘polyclip’, ‘FNN’, ‘dqrng’, ‘SeuratObject’, ‘cowplot’, ‘fastDummies’, ‘fitdistrplus’, ‘future’, ‘future.apply’, ‘ggrepel’, ‘ggridges’, ‘ica’, ‘igraph’, ‘irlba’, ‘lmtest’, ‘matrixStats’, ‘patchwork’, ‘pbapply’, ‘plotly’, ‘png’, ‘progressr’, ‘RANN’, ‘RcppAnnoy’, ‘RcppHNSW’, ‘reticulate’, ‘ROCR’, ‘RSpectra’, ‘Rtsne’, ‘scattermore’, ‘sctransform’, ‘spatstat.explore’, ‘spatstat.geom’, ‘uwot’, ‘RcppEigen’, ‘RcppProgress’


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

'getOption("repos")' replac

ERROR: Error in Read10X_h5("scRNAseq/GSM4339769_C141_filtered_feature_bc_matrix.h5"): Please install hdf5r to read HDF5 files


---
# 1️⃣ Quality control  ·  ~3 min

Real droplets contain empty beads, dying cells, and doublets. We flag low-quality cells using three metrics per cell: number of genes (`nFeature_RNA`), total UMIs (`nCount_RNA`), and the fraction of reads from mitochondrial genes (`percent.mt` — high values signal stressed or dying cells).

*The Seurat object and `percent.mt` are already built in setup. Here we just look, then filter.*

**Distributions across all cells** — one violin per metric:

In [ ]:
#This function is used to create violin plots, which display the distribution of values ​​for a given feature across cells in a dataset.
scPlot <- VlnPlot(sampleA, features = c("nFeature_RNA", "nCount_RNA", "percent.mt"), ncol=3)
scPlot
#ggsave("01-VlnPlot.png",plot = scPlot, bg = 'white')

ERROR: Error: object 'sampleA' not found


Metrics are more informative viewed **together**. UMIs vs. mitochondrial % separates healthy cells (bottom) from dying ones (top):

In [ ]:
scPlot <- FeatureScatter(sampleA, feature1="nCount_RNA", feature2="percent.mt")
scPlot
#ggsave("08-FeatureScatter.png",plot = scPlot, bg = 'white')

**Apply the filter.** Keep cells with >500 UMIs, <7000 genes (drops likely doublets), and <10% mitochondrial reads:

In [ ]:
before <- ncol(sampleA)

sampleA <- subset(sampleA,
                  nCount_RNA > 500 &     # cells with more than 500 UMIs
                  nFeature_RNA < 7000 &  # fewer than 7000 genes detected
                  percent.mt < 10)       # less than 10% mitochondrial reads

cat("Cells before:", before, " -> after:", ncol(sampleA), "\n")
# (Object was already filtered in setup, so this is a no-op on counts —
#  it shows the audience exactly which thresholds define a 'good' cell.)


---
# 2️⃣ Clustering  ·  ~4 min

After normalization → variable-gene selection → PCA → UMAP (all done in setup), cells that express similar genes sit near each other. Clustering formalizes those neighborhoods into discrete groups.

*Heavy steps are pre-computed; the cells below draw from the finished object.*

The UMAP embedding — each dot is a cell, positioned by overall expression similarity:

In [ ]:
scPlot <- DimPlot(sampleA, reduction = "umap") + NoLegend()
scPlot


Now color by **cluster**. A low resolution (`0.01`) gives a handful of clean, well-separated groups — the right granularity for a first pass:

In [ ]:
# clusters already computed in setup (resolution 0.01) — just color the UMAP by them
scPlot <- DimPlot(sampleA, reduction = "umap", label = TRUE) + NoLegend()
scPlot


**What defines each cluster?** `FindAllMarkers` (pre-computed in setup) finds genes over-expressed in each group. The heatmap shows the top 5 markers per cluster — the diagonal blocks confirm the clusters are transcriptionally distinct:

In [ ]:
# markers were computed in setup; just pick the top 5 per cluster and plot
topMarkers <- sampleA.markers %>%
  group_by(cluster) %>%
  top_n(n = 5, wt = avg_log2FC)

scPlot <- DoHeatmap(sampleA, features = topMarkers$gene) + NoLegend()
scPlot


---
# 3️⃣ Cell-type annotation  ·  ~4 min

Clusters are just numbers until we assign biological identities. Two complementary approaches:

- **Automated** — `SingleR` scores every cell against a labeled reference (Blueprint/ENCODE).
- **Manual** — check expression of canonical marker genes we already trust.

*`SingleR` (`pred`) and the reference are pre-computed in setup.*

**Automated:** SingleR's per-cell scores against each reference cell type. Bright bands = confident calls:

In [ ]:
#png("52-plotScoreHeatmap.png", res = 300, width = 1920, height = 1920)
plotScoreHeatmap(pred)
#dev.off()

Which SingleR label lands in which Seurat cluster — a contingency table connecting the two approaches:

In [ ]:
# Create a contingency table comparing assigned labels to Seurat clusters
my.table <- table(Assigned = pred$pruned.labels,
                  cluster = my.sce$seurat_clusters)
my.table

# Plot heatmap of the contingency table
library(pheatmap)
pheatmap(log2(my.table + 1), filename = "53-pheatmap.png")

**Manual:** a curated immune-marker panel. The dotplot shows, per cluster, what fraction of cells express each marker (dot size) and how strongly (color). Clusters light up for Myeloid, T, B, NK, DC, etc. — confirming this BALF sample is dominated by immune cells:

In [ ]:
# Select a set of marker genes for immune cells
genes_markers = list(
  Myeloid = c(
    "LYZ","S100A8","S100A9","LGALS3","CTSS","MS4A7","LST1"
  ),
  Lymph_T = c(
    "CD3D","CD3E","TRAC","TRBC1","IL7R","LTB"
  ),
  Lymph_B = c(
    "MS4A1","CD79A","CD79B","CD74","HLA-DRA","CD37","CD19"
  ),
  Plasma = c(
    "MZB1","XBP1","JCHAIN","SDC1","TNFRSF17","IGKC","IGHG1"
  ),
  DC = c(
    "FCER1A","CLEC10A","CD1C","ITGAX","LILRA4","IRF7"
  ),
  NK = c(
    "NKG7","KLRD1","GNLY","PRF1","TRDC"
  ),
  Erythrocytes = c(
    "HBB","HBA1","HBA2","ALAS2","SLC4A1","AHSP","GYPA"
  )
)

# Note that many of these markers may not appear in our datasets, however
# the combination of these markers allows for more safe annotation.

# Plot figure
scPlot <- DotPlot(
  object   = sampleA,
  features = genes_markers,
  group.by = "seurat_clusters",
)

scPlot$data = scPlot$data %>% # Filter out genes with more than 5% in each cluster
  dplyr::filter(pct.exp >= 5)

scPlot = scPlot +
  scale_color_gradient2(
  low = "#2c7bb6",
  mid = "#b2182b",  # Assign a color scale to the dotplot in different levels
  high = "#d7191c") +
  theme_bw(base_size = 12) + # Change non-data graph display
  theme(
    axis.text.x  = element_text(angle = 90) # Change axis x angle
  )

options(repr.plot.width=14, repr.plot.height=8)
scPlot

#ggsave("dotplot_immuneMarkers_facets.png", width = 23, plot = scPlot, bg = 'white')

---
# 4️⃣ Functional analysis — GO enrichment  ·  ~3 min

Finally, we ask *what the marker genes are doing biologically*. GO over-representation analysis tests whether the cluster markers fall into particular **Biological Process** categories more than expected by chance.

*Gene-ID mapping (`my.map`) is pre-computed in setup.*

The filtered, ID-mapped marker genes we'll test (SYMBOL → ENTREZID was done in setup):

In [ ]:
head(my.map)
cat("Genes tested:", nrow(my.map), "\n")


**Run GO over-representation analysis** (Biological Process, BH-adjusted):

In [ ]:
# This function performs GO enrichment analysis on a given set of genes.
ego <- enrichGO(gene   = my.map$ENTREZID, # Specifies the list of gene ENTREZ ID to be grouped
                OrgDb  = org.Hs.eg.db, # Specifies the organism database for human genes
                ont    = "BP", # Specifies that the GO grouping should be based on the "Biological Process" subontology.
                pAdjustMethod = "BH", # The method used for adjusting p-values to control the false discovery rate of Benjamini-Hochberg (BH)
                pvalueCutoff  = 0.01, # The p-value threshold for significance
                qvalueCutoff  = 0.05, # The q-value threshold for significance
                readable      = TRUE) # Converts the Entrez IDs to gene symbols for easier interpretation.

# displays the first few entries of the ego object
head(ego)

Visualize the top enriched terms:

In [ ]:
# barplot of the most significantly enriched GO Biological Process terms
library(enrichplot)
barplot(ego, showCategory = 12) + ggtitle("GO: Biological Process (cluster markers)")


---
## ✅ Recap

In ~15 minutes we went from a raw count matrix to biological interpretation:

| Step | What we did | Key output |
|------|-------------|------------|
| **QC** | Filtered on genes, UMIs, mito % | Clean cell set |
| **Clustering** | UMAP + Leiden at low resolution | A few distinct clusters + marker heatmap |
| **Annotation** | SingleR + manual marker dotplot | Immune cell-type labels |
| **Function** | GO over-representation of markers | Enriched biological processes |

The full module additionally covers doublet detection, SCTransform, resolution sweeps (`clustree`), differential abundance (Milo), and ambient-RNA correction (SoupX) — omitted here for time.